# SocialPulse AI - Exploratory Data Analysis (EDA)
## Module 1: Post Engagement & Virality Prediction

This notebook conducts an in-depth exploratory analysis on social media posts and engagement patterns. Key focus areas:
1. **Target Distribution**: Proportions of Low (50%), Moderate (35%), and Viral (15%) posts.
2. **Temporal Dynamics**: Posting time, peak traffic windows, and weekend variance.
3. **Content Attributes**: Hashtags, text length, question hooks, and emojis.
4. **Multimodal Signals**: Media attachment impact (Video vs Image vs Text-only).
5. **Account Reach & Sentiment**: Log-followers distribution and VADER sentiment correlation.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
sys.path.append(str(Path.cwd().parent))
import config

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
print('Libraries imported successfully!')

### 1. Load Processed Dataset

In [2]:
df = pd.read_csv(config.PROCESSED_DATASET_PATH)
print(f'Dataset Shape: {df.shape}')
df.head(3)

### 2. Target Class Distribution
Verifying engagement class percentages (Low 50%, Moderate 35%, Viral 15%).

In [3]:
class_counts = df['engagement_class'].value_counts()[['Low', 'Moderate', 'Viral']]
class_pcts = df['engagement_class'].value_counts(normalize=True)[['Low', 'Moderate', 'Viral']] * 100

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#64748B', '#3B82F6', '#F59E0B']

# Bar plot
bars = ax[0].bar(class_counts.index, class_counts.values, color=colors, edgecolor='#333333', linewidth=1)
ax[0].set_title('Target Class Distribution (Post Counts)', fontsize=13, weight='bold')
ax[0].set_ylabel('Number of Posts', fontsize=11)
for bar in bars:
    h = bar.get_height()
    ax[0].annotate(f'{h:,}', xy=(bar.get_x() + bar.get_width()/2, h), xytext=(0, 4),
                   textcoords='offset points', ha='center', va='bottom', weight='bold')

# Donut chart
wedges, texts, autotexts = ax[1].pie(class_pcts, labels=class_pcts.index, autopct='%1.1f%%',
                                      colors=colors, startangle=140, pctdistance=0.75,
                                      wedgeprops=dict(width=0.45, edgecolor='#333333'))
for at in autotexts:
    at.set_weight('bold')
ax[1].set_title('Target Engagement Share (%)', fontsize=13, weight='bold')

plt.tight_layout()
plt.show()

### 3. Engagement vs. Posting Hour (Temporal Analysis)
Examining when posts achieve highest virality rates across the 24-hour cycle.

In [4]:
df['dt'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['dt'].dt.hour

hour_summary = df.groupby('hour')['engagement_score'].mean().reset_index()

plt.figure(figsize=(12, 5))
sns.lineplot(data=hour_summary, x='hour', y='engagement_score', marker='o', color='#2563EB', lw=2.5)
plt.axvspan(12, 14, color='#10B981', alpha=0.15, label='Lunch Peak (12-14h)')
plt.axvspan(18, 22, color='#F59E0B', alpha=0.15, label='Evening Prime Peak (18-22h)')
plt.title('Average Engagement Score vs. Hour of Day', fontsize=13, weight='bold')
plt.xlabel('Hour of Day (0-23)', fontsize=11)
plt.ylabel('Mean Engagement Score (per 1k followers)', fontsize=11)
plt.xticks(range(0, 24))
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

### 4. Media Type vs. Virality
Analyzing the impact of video and image attachments compared to text-only posts.

In [5]:
media_cross = pd.crosstab(df['media_type'], df['engagement_class'], normalize='index')[['Low', 'Moderate', 'Viral']] * 100

media_cross.plot(kind='bar', stacked=True, color=['#64748B', '#3B82F6', '#F59E0B'], figsize=(9, 5), edgecolor='#333333')
plt.title('Engagement Class Composition by Media Type (%)', fontsize=13, weight='bold')
plt.xlabel('Media Attachment Type', fontsize=11)
plt.ylabel('Percentage of Posts', fontsize=11)
plt.legend(title='Engagement Tier', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 5. Hashtag Count vs. Engagement
Identifying the optimal hashtag volume for discoverability without triggering spam penalties.

In [6]:
from src.features import extract_text_statistics
df['hashtag_cnt'] = df['text'].apply(lambda t: extract_text_statistics(t)['hashtag_count'])

tag_summary = df.groupby('hashtag_cnt')['engagement_score'].mean().reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=tag_summary, x='hashtag_cnt', y='engagement_score', color='#8B5CF6', edgecolor='#333333')
plt.title('Hashtag Count vs. Mean Engagement Score', fontsize=13, weight='bold')
plt.xlabel('Number of Hashtags in Post', fontsize=11)
plt.ylabel('Mean Engagement Score', fontsize=11)
plt.tight_layout()
plt.show()

### 6. Follower Distribution (Log Scale)
Visualizing the account reach distribution across the dataset.

In [7]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['followers'], bins=40, kde=True, color='#0284C7', ax=ax[0])
ax[0].set_title('Raw Follower Count Distribution (Skewed)', fontsize=12, weight='bold')
ax[0].set_xlabel('Followers', fontsize=11)

sns.histplot(np.log1p(df['followers']), bins=40, kde=True, color='#059669', ax=ax[1])
ax[1].set_title('Log-Transformed Followers (Normalized)', fontsize=12, weight='bold')
ax[1].set_xlabel('log(Followers + 1)', fontsize=11)

plt.tight_layout()
plt.show()

### 7. Sentiment Polarity Across Engagement Tiers

In [8]:
from src.features import extract_sentiment_scores
df['sentiment_compound'] = df['text'].apply(lambda t: extract_sentiment_scores(t)['vader_compound'])

plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='engagement_class', y='sentiment_compound', order=['Low', 'Moderate', 'Viral'], palette=['#64748B', '#3B82F6', '#F59E0B'])
plt.title('VADER Sentiment Polarity by Engagement Tier', fontsize=13, weight='bold')
plt.xlabel('Engagement Tier', fontsize=11)
plt.ylabel('VADER Compound Score (-1 to +1)', fontsize=11)
plt.tight_layout()
plt.show()